In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import mlflow
import mlflow.pytorch

# 0. SAMBUNGKAN KE SERVER (Ini yang sempat terlupa setelah kernel di-restart!)
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("cifar10-experiment")

# 1. Siapkan Data Fashion-MNIST (Ukuran Kecil & Cepat)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

valset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
valloader = torch.utils.data.DataLoader(valset, batch_size=64, shuffle=False)

# 2. Buat Arsitektur Model CNN
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1) 
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Mulai Eksperimen dan Tracking dengan MLflow
with mlflow.start_run(run_name="baseline-cnn-fashion"):
    
    mlflow.log_param("dataset", "Fashion-MNIST")
    mlflow.log_param("learning_rate", 0.001)
    mlflow.log_param("epochs", 3)
    mlflow.log_param("optimizer", "Adam")

    print("Mulai training model...")
    epochs = 3
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in trainloader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        train_loss = running_loss / len(trainloader)
        
        # Validasi
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in valloader:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        val_loss = val_loss / len(valloader)
        val_acc = correct / total
        
        mlflow.log_metric("train_loss", train_loss, step=epoch)
        mlflow.log_metric("val_loss", val_loss, step=epoch)
        mlflow.log_metric("val_acc", val_acc, step=epoch)
        
        print(f"Epoch {epoch+1}/{epochs} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Tambahan format Pickle agar tidak error
    mlflow.pytorch.log_model(model, "model", serialization_format="pickle")
    print("Selesai! Berhasil dicatat ke MLflow.")

2026/10/11 08:20:08 INFO mlflow.tracking.fluent: Experiment with name 'cifar10-experiment' does not exist. Creating a new experiment.


Mulai training model...
Epoch 1/3 | Val Loss: 0.3683 | Val Acc: 0.8669
Epoch 2/3 | Val Loss: 0.3153 | Val Acc: 0.8858


2026/10/11 08:21:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Epoch 3/3 | Val Loss: 0.2790 | Val Acc: 0.8959


2026/10/11 08:22:00 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


Selesai! Berhasil dicatat ke MLflow.
🏃 View run baseline-cnn-fashion at: http://localhost:5000/#/experiments/1/runs/e7a911e567f94b9882fa0390b0eb31e1
🧪 View experiment at: http://localhost:5000/#/experiments/1
